# Variant A — fully nested participant-level evaluation

Only Variant A is trained. Configuration selection uses three inner participant folds, with an additional disjoint early-stopping split inside each inner training set. LP coefficients/scales use **only core training controls**. After selecting configuration and epochs entirely within outer training, the model and LP are refitted on all outer-training participants and evaluated once on the outer test fold.

## Compatibility and safety update
- Registered attention-pooling layers replace unsafe/shape-ambiguous Lambdas.
- Unique search configurations and inner folds are fixed independently of training seed.
- Curated checksum-verified cohort is the default: 161 participants, 292 recordings. Clinical metadata are joined by exact verified IDs and are diagnostics only.
- Bounded-memory batch training works with modern Keras without obsolete worker arguments.
- Atomic checkpoints after every completed epoch retain model, optimizer, LR/stopping state and history. Inner assessments and outer tasks also commit independently.
- `RUN_EXPERIMENT=False` initially. `MAX_NEW_TASKS=1` allows incremental outer-task execution; repeat the runner to continue.

Use a **new** versioned checkpoint directory: the corrected reference-fitting and candidate protocol must not reuse older CSV-only outputs. Set a persistent path or explicitly export/download the checkpoint ZIP before a Kaggle/Colab runtime reset. Existing downloaded results are not overwritten.

In [ ]:
# Run this setup first in a fresh Kaggle/Colab kernel.
import os, sys, tempfile, subprocess
from pathlib import Path
from collections import Counter
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH','true')
SOURCE_REV = '5c8bd0df73870c03baf9d0f7252a7eeb73131328'
REPO_URL = 'https://github.com/AlexM3020/Subject-Independent-Evaluation-of-Mamba-for-Parkinson-s-Disease-Detection-from-Gait-Signals.git'
INSTALL_DEPENDENCIES = False
roots=[Path.cwd(),*Path.cwd().parents]
REPO_ROOT=next((p for b in roots for p in (b,b/'paper_repository') if (p/'gait/tf_notebook_support.py').is_file()),None)
if REPO_ROOT is None:
    REPO_ROOT=Path(tempfile.gettempdir())/('gait-keras-'+SOURCE_REV[:12])
    if not REPO_ROOT.exists(): subprocess.run(['git','clone','--no-checkout',REPO_URL,str(REPO_ROOT)],check=True)
    subprocess.run(['git','-C',str(REPO_ROOT),'checkout','--detach',SOURCE_REV],check=True)
REPO_ROOT=REPO_ROOT.resolve()
for filename in ('__init__.py','official_data.py','tf_notebook_support.py'):
    expected=subprocess.run(['git','-C',str(REPO_ROOT),'show',SOURCE_REV+':gait/'+filename],capture_output=True,check=True).stdout
    assert (REPO_ROOT/'gait'/filename).read_bytes().replace(b'\r\n',b'\n')==expected.replace(b'\r\n',b'\n'), 'Pinned source changed: '+filename
stale=[name for name,module in list(sys.modules.items()) if (name=='gait' or name.startswith('gait.')) and getattr(module,'__file__',None) and not Path(module.__file__).resolve().is_relative_to(REPO_ROOT)]
if stale: raise RuntimeError('Older gait imports cached: export needed results, restart kernel and run this setup first. '+str(stale))
sys.path.insert(0,str(REPO_ROOT))
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO_ROOT/'requirements-tensorflow.txt')],check=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
keras=tf.keras
from scipy.signal import lfilter
from sklearn.metrics import (roc_auc_score,accuracy_score,balanced_accuracy_score,f1_score,confusion_matrix,ConfusionMatrixDisplay,log_loss,roc_curve)
from IPython.display import display
from gait import tf_notebook_support as nb
FAMILY='A'
SEED,FOLD_SEED,SPLIT_SEED,SEARCH_SEED=1,0,9137,20261117
N_FOLDS,N_INNER,VAL_FRAC=10,3,.15
LP_ORDER,WIN,STRIDE=11,100,50
BATCH_WIN,BATCH_REC=128,64
LR_WIN,LR_REC,LABEL_SMOOTH,L2_REG=5e-4,1e-3,.1,1e-3
MAX_EPOCHS_WIN,MAX_EPOCHS_REC=1000,200
PATIENCE,RLR_PATIENCE,MIN_LR=25,14,1e-7
THRESHOLD=.5
WORK_ROOT=Path('/kaggle/working') if Path('/kaggle').exists() else Path('/content') if Path('/content').exists() else Path.cwd()
CHECKPOINT_DIR=WORK_ROOT/'lpgnet_variant_a_keras3_v2'
OUTDIR=CHECKPOINT_DIR/'reports'
CACHE_DIR=Path(tempfile.gettempdir())/'gaitpdb-1.0.0-verified-cache'
COHORT_POLICY='curated'
ALLOW_DOWNLOAD=True
RUN_EXPERIMENT=False
MAX_NEW_TASKS=1
VERBOSE_EPOCHS=1
RESTORE_FROM_ZIP=''
EXPERIMENT_COMPLETE=False
summary=participant_ci=participant_fold_metrics=selection_table=trial_table=nested_oof=pd.DataFrame()
participants=participant_errors=pd_severity=by_study=seed_stability=inner_loss=pd.DataFrame()
keras.utils.set_random_seed(SEED)
print('Pinned source:',SOURCE_REV,'| TensorFlow:',tf.__version__,'| Keras:',keras.__version__)
print('GPU:',tf.config.list_physical_devices('GPU'))
print('Checkpoint directory:',CHECKPOINT_DIR)
print('For cross-session safety, export checkpoint bundles or set CHECKPOINT_DIR to mounted persistent storage before starting.')

In [ ]:
TRAIN_SEEDS = [42]  # Declare [42,123,2026] before starting for a multi-seed experiment.
N_TRIALS = 12
N_BOOT = 1000
print('Variant A only; no baseline or GRU training.')
print('Outer tasks:',len(TRAIN_SEEDS)*N_FOLDS,'| inner two-stage fits:',len(TRAIN_SEEDS)*N_FOLDS*N_TRIALS*N_INNER)
print('Configuration and fold assignments are frozen in the checkpoint plan.')

In [ ]:
records,clinical,data_audit,data_provenance = nb.load_inputs(CACHE_DIR,COHORT_POLICY,ALLOW_DOWNLOAD)
subjects=sorted({r['subject'] for r in records})
subj_label={r['subject']:r['label'] for r in records}
by_file={r['file']:i for i,r in enumerate(records)}
by_subject={s:[i for i,r in enumerate(records) if r['subject']==s] for s in subjects}
print('Cohort policy:',COHORT_POLICY)
print(len(records),'recordings |',len(subjects),'participants |',dict(Counter(subj_label.values())))
print('Data provenance:',data_provenance)
print('Clinical unknowns are retained; no severity is inferred from filenames.')

In [ ]:
def preprocess(x):
    return lfilter([.5,.5],1,x,axis=0)[::2]

def fit_lp(control_sigs,order=LP_ORDER):
    return nb.fit_lp(control_sigs,order)

def apply_lp(x,C,stds):
    return nb.apply_lp(x,C,stds)

def lp_features(indices,C,stds):
    return {i:apply_lp(records[i]['proc'],C,stds) for i in indices}

for r in records:
    r['proc']=preprocess(r['signal'])
print('Only downsampling is prepared globally. Early-stopping and inner-assessment controls never fit the selection-stage LP reference.')

In [ ]:
def make_folds():
    return nb.outer_folds(records,N_FOLDS,FOLD_SEED)

def split_val(indices,frac=VAL_FRAC,seed=SEED):
    return nb.subject_split(records,indices,frac,seed)

def labels(indices):
    return np.array([records[i]['label'] for i in indices],np.float32)

FOLDS=make_folds()
for k,f in enumerate(FOLDS):
    print('Fold',k,'train participants:',len({records[i]['subject'] for i in f['train']}),
          'test participants:',len({records[i]['subject'] for i in f['test']}))

In [ ]:
# Shared Keras3-compatible training support; no materialization of all training windows.
# Models are checkpointed with optimizer state after each complete epoch.
# EarlyStopping/LR counters and absolute-epoch batch order are preserved.
build_recording_model=nb.build_recording_model
print('Epoch recovery uses committed .keras models plus atomic JSON state, not CSV row counts.')
print('Old CSV-only checkpoint directories are intentionally not imported into this corrected protocol.')

In [ ]:
# ============================================================
# OOF metrics, participant aggregation, errors, and clinical subgroups
# ============================================================
def _validated_binary_inputs(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    if y.ndim != 1 or p.ndim != 1 or len(y) == 0 or len(y) != len(p):
        raise ValueError('Labels and probabilities must be nonempty 1-D arrays of equal length.')
    if np.iscomplexobj(y) or not np.isin(y, [0, 1]).all():
        raise ValueError('Labels must contain only binary values 0 and 1.')
    if np.iscomplexobj(p):
        raise ValueError('Probabilities must be real numbers in [0, 1].')
    p = np.asarray(p, float)
    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError('Probabilities must be finite and in [0, 1].')
    return y.astype(int), p

def metric_dict(y, p):
    y, p = _validated_binary_inputs(y, p)
    yh = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0,1]).ravel()
    return dict(n=len(y), accuracy=accuracy_score(y, yh),
                balanced_accuracy=balanced_accuracy_score(y, yh),
                macro_f1=f1_score(y, yh, labels=[0,1], average='macro', zero_division=0),
                f1_pd=f1_score(y, yh, zero_division=0),
                sensitivity=tp/(tp+fn) if tp+fn else np.nan,
                specificity=tn/(tn+fp) if tn+fp else np.nan,
                roc_auc=roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan,
                log_loss=log_loss(y, np.clip(p,1e-7,1-1e-7), labels=[0,1]),
                brier=float(np.mean((p-y)**2)))

def oof_frame(rows):
    df = pd.DataFrame(rows)
    y, p = _validated_binary_inputs(df['label'], df['probability'])
    df['label'] = y; df['probability'] = p
    df['prediction'] = (df['probability'] >= 0.5).astype(int)
    df['correct'] = (df['prediction'] == df['label']).astype(int)
    df['confidence'] = np.maximum(df['probability'], 1-df['probability'])
    df['nll'] = -(df['label']*np.log(np.clip(df['probability'],1e-7,1)) +
                  (1-df['label'])*np.log(np.clip(1-df['probability'],1e-7,1)))
    # Strict metadata lookup: missing subjects or fields raise KeyError.
    for col in ('study','diagnosis','hy','hy_label','hy_status','age','gender'):
        df[col] = df['subject'].map(lambda s: clinical[s][col])
    return df

def participant_oof(df, group_cols):
    """Equal weight per participant; probability is averaged over that person's walks."""
    keys = group_cols + ['subject']
    df = df.copy()
    y, p = _validated_binary_inputs(df['label'], df['probability'])
    df['label'] = y; df['probability'] = p
    if df[keys + ['fold']].isna().any().any():
        raise ValueError('Participant grouping keys and folds must not be missing.')
    grouped = df.groupby(keys, as_index=False)
    counts = grouped[['label', 'fold']].nunique(dropna=False)
    if (counts['label'] > 1).any():
        raise ValueError('Conflicting labels for a subject within an evaluation group.')
    if (counts['fold'] > 1).any():
        raise ValueError('Multiple folds for a subject within an evaluation group.')
    if 'file' in df.columns and df.duplicated(subset=group_cols + ['file']).any():
        raise ValueError('Duplicate file within an evaluation group.')
    agg = dict(label=('label','first'), fold=('fold','first'), probability=('probability','mean'),
               recordings=('file','nunique') if 'file' in df.columns else ('subject','size'))
    out = grouped.agg(**agg)
    return oof_frame(out.to_dict('records'))

def summarize_oof(df, group_cols):
    rows = []
    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols, keys))
        rec = metric_dict(g['label'], g['probability'])
        part = participant_oof(g, group_cols)
        pm = metric_dict(part['label'], part['probability'])
        rows.append({**base, **{'recording_'+k:v for k,v in rec.items()}})
        rows[-1].update({'participant_'+k:v for k,v in pm.items()})
    return pd.DataFrame(rows)

def wilson_interval(correct, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph = correct/n; den = 1+z*z/n
    center = (ph+z*z/(2*n))/den
    half = z*np.sqrt(ph*(1-ph)/n+z*z/(4*n*n))/den
    return center-half, center+half

def error_tables(df, group_cols):
    part = participant_oof(df, group_cols)
    errors = part[part['correct'] == 0].sort_values(group_cols+['nll'], ascending=[True]*len(group_cols)+[False])
    pd_rows = []
    for keys, g in part[part['label'] == 1].groupby(group_cols+['hy_label'], dropna=False):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols+['hy_label'], keys))
        n = len(g); correct = int(g['correct'].sum()); lo, hi = wilson_interval(correct, n)
        pd_rows.append({**base, 'pd_participants':n, 'correct':correct, 'errors':n-correct,
                        'pd_sensitivity':correct/n, 'wilson_95_low':lo, 'wilson_95_high':hi})
    study_rows = []
    for keys, g in part.groupby(group_cols+['study','diagnosis']):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols+['study','diagnosis'], keys))
        study_rows.append({**base, **metric_dict(g['label'], g['probability'])})
    return part, errors, pd.DataFrame(pd_rows), pd.DataFrame(study_rows)

def show_table(df, title, limit=100):
    print('\n'+title)
    display(df.head(limit))

def pooled_bootstrap(df, group_cols, metrics=('accuracy','balanced_accuracy','macro_f1','sensitivity','specificity','roc_auc')):
    """Stratified participant bootstrap conditional on fitted OOF predictions."""
    rows = []
    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols, keys))
        part = participant_oof(g, group_cols).reset_index(drop=True)
        y = part['label'].to_numpy(); p = part['probability'].to_numpy()
        class_idx = [np.flatnonzero(y == c) for c in (0,1)]
        rng = np.random.default_rng(2026)
        for metric in metrics:
            observed = metric_dict(y,p)[metric]
            vals = []
            for _ in range(N_BOOT):
                ii = np.concatenate([rng.choice(ix, len(ix), replace=True) for ix in class_idx])
                vals.append(metric_dict(y[ii],p[ii])[metric])
            rows.append({**base,'metric':metric,'estimate':observed,
                         'bootstrap_95_low':np.nanpercentile(vals,2.5),
                         'bootstrap_95_high':np.nanpercentile(vals,97.5)})
    return pd.DataFrame(rows)

def plot_confusion_and_roc(df, group_cols, title_col):
    part = participant_oof(df, group_cols)
    groups = list(part.groupby(group_cols))
    fig, axes = plt.subplots(len(groups), 2, figsize=(11,4.3*len(groups)), squeeze=False)
    for row, (keys,g) in enumerate(groups):
        if not isinstance(keys, tuple): keys=(keys,)
        title = ' | '.join(f'{c}={v}' for c,v in zip(group_cols,keys))
        cm = confusion_matrix(g['label'],g['prediction'],labels=[0,1])
        ConfusionMatrixDisplay(cm,display_labels=['Control','PD']).plot(ax=axes[row,0],colorbar=False)
        axes[row,0].set_title(title+' | participant OOF')
        fpr,tpr,_ = roc_curve(g['label'],g['probability'])
        axes[row,1].plot(fpr,tpr,label=f"AUC={roc_auc_score(g['label'],g['probability']):.3f}")
        axes[row,1].plot([0,1],[0,1],'--',color='gray')
        axes[row,1].set(xlabel='False-positive rate',ylabel='True-positive rate',title=title)
        axes[row,1].legend()
    fig.tight_layout(); plt.show()


In [ ]:
def build_variantA(**kwargs):
    return nb.build_variantA(**kwargs)

SPACE_A=dict(channels=[32,48],kernels=[(7,5,3),(9,7,5)],se_ratio=[4,8],
             spat_drop=[.1,.2,.3],drop=[.25,.4],l2=[1e-4,1e-3])

def parameter_report(cfg):
    return nb.model_parameter_report(FAMILY,cfg,WIN,LP_ORDER)

MODEL_READY=False
smoke_cfg={k:v[0] for k,v in SPACE_A.items()}
print('Actual forward/backward and safe model/optimizer reload:',nb.smoke_check(FAMILY,smoke_cfg,WIN))
MODEL_READY=True
print('Attention pooling uses a registered WeightedSum layer instead of an anonymous Lambda.')

In [ ]:
# Corrected selection: early-stopping controls are excluded from LP fitting.
# This applies both to inner configuration assessment and outer-development epoch selection.
# Only after selection are LP and both stages refitted on all outer-training participants.
# Candidate lists and inner partitions are independent of training seed.
preview=nb.unique_candidates(SPACE_A,N_TRIALS,SEARCH_SEED)
show_table(pd.DataFrame(preview),'Outer-fold-0 unique candidate preview (not model results)')
print('This protocol change requires a new checkpoint directory; old CSV-only checkpoints cannot be mixed in.')

In [ ]:
if RUN_EXPERIMENT:
    assert MODEL_READY,'Run the real model/checkpoint smoke check first'
    if RESTORE_FROM_ZIP:
        print('Restored:',nb.restore_checkpoint_bundle(RESTORE_FROM_ZIP,CHECKPOINT_DIR))
        RESTORE_FROM_ZIP=''
    experiment=nb.NotebookExperiment(globals(),FAMILY)
    run_result=experiment.nested(max_tasks=MAX_NEW_TASKS)
    globals().update(nb.unpack_result(globals(),run_result,FAMILY))
    print('Completed outer tasks:',len(run_result['selections']),'/',len(TRAIN_SEEDS)*N_FOLDS)
    print('Experiment complete:',EXPERIMENT_COMPLETE)
else:
    EXPERIMENT_COMPLETE=False
    print('Set RUN_EXPERIMENT=True to start/resume. Epoch and task checkpoints are automatic once training starts.')

In [ ]:
if EXPERIMENT_COMPLETE:
    globals().update(nb.summary_tables(globals(),FAMILY))
    show_table(summary.round(4),'Complete pooled recording and participant OOF metrics by seed')
    show_table(participant_ci.round(4),'Participant-stratified bootstrap intervals; conditional on fitted predictions')
    show_table(participant_fold_metrics.round(4),'Participant fold metrics; fold SD is descriptive')
    show_table(selection_table,'Selected Variant-A configurations',limit=100)
    show_table(trial_table,'All inner candidate assessments',limit=100)
else:
    print('Complete all outer tasks before inspecting test performance.')

In [ ]:
if EXPERIMENT_COMPLETE:
    globals().update(nb.error_reports(globals(),FAMILY))
    show_table(participants.sort_values('nll',ascending=False).round(4),'Complete participant table, highest individual NLL first',limit=200)
    show_table(participant_errors.round(4),'Misclassified participants',limit=200)
    show_table(pd_severity.round(4),'PD sensitivity by verified HY stage')
    show_table(by_study.round(4),'Study-by-diagnosis metrics; single-class AUC remains undefined')
    show_table(seed_stability.round(4),'Across-seed errors and probability variability')
    show_table(inner_loss.head(50).round(4),'Selected inner-assessment NLL contributors; repetitions across outer folds are not independent')
else:
    print('Error diagnostics wait for the complete experiment.')

In [ ]:
EXPORT_CHECKPOINT_BUNDLE=False
EXPORT_ANALYSIS_CSV=False
if EXPORT_CHECKPOINT_BUNDLE:
    print('Checkpoint ZIP:',nb.export_checkpoint_bundle(CHECKPOINT_DIR))
    print('Download or retain on persistent storage before ending the runtime.')
if EXPORT_ANALYSIS_CSV:
    if not EXPERIMENT_COMPLETE: raise RuntimeError('Analysis export requires complete OOF results')
    nb.export_tables(globals(),FAMILY)
if not EXPORT_CHECKPOINT_BUNDLE and not EXPORT_ANALYSIS_CSV:
    print('No extra export requested. Automatic epoch/task checkpoints remain in',CHECKPOINT_DIR)

## Reporting checklist

Report all outer folds/seeds, the candidate budget, curated-cohort exclusions and verified clinical coverage. Nested assessment is distinct from early stopping: neither early-stopping nor inner-assessment controls fit the selection-stage LP reference. The final fixed-epoch refit uses all outer-training participants only.

The Keras3 version deliberately fixes the earlier early-control preprocessing exposure and seed-dependent search partitions; use a new run directory and do not combine its OOF rows with old results. Record the modern separable-convolution regularizer behavior rather than claiming bit-identical historical training.

Checkpoint state is authoritative, not exported CSV row counts. Atomic checkpoints recover completed epochs and completed nested tasks; stochastic numerical identity across different GPU runtimes is not guaranteed. Kernel restarts and full runtime resets are different: use an exported full checkpoint bundle or mounted persistent storage for the latter.

Participant bootstrap intervals condition on fitted OOF predictions, not new clinics or model-refitting uncertainty. HY and study subgroup results remain descriptive. The update includes executable smoke checks, not new full-cohort measurements.